- identify env parameters from data
- estimate the parameter uncertainty
    - Do multiple runs and consider the variance?

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"]="false"
from functools import partial
import time
from tqdm import tqdm
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
import jax
import jax.numpy as jnp
# jax.config.update("jax_enable_x64", True)
gpus = jax.devices()
print(gpus)

jax.config.update("jax_default_device", gpus[0])

import diffrax
import equinox as eqx
import optax

from haiku import PRNGSequence

import exciting_environments as excenvs

from dmpe.data_management import DataPaths
from dmpe.models.models import NeuralEulerODECartpole
from dmpe.evaluation.experiment_utils import get_experiment_ids, load_experiment_results, load_all_experiment_results
from dmpe.evaluation.data_evaluation import DataEvaluator, JensenShannonDivergence
from dmpe.evaluation.model_evaluation import ModelWrapper, ModelEvaluator, PredictionComparison, NodeModelWrapper, EnvWrapper
from dmpe.evaluation.utils import default_constraint_function

from dmpe.utils.env_utils.cart_pole_utils import setup_env as setup_cart_pole_env

In [ ]:
from dmpe.models.model_training import ModelTrainer
from dmpe.evaluation.exp_data_model_learning import train_model_on_experiment_data, ModelExpDataResult

In [ ]:
env, _ = setup_cart_pole_env()

In [ ]:
for init_omega in jnp.arange(-1, 1, 0.1):
    init_obs = jnp.array([0, 0, 0., init_omega])
    init_state = env.generate_state_from_observation(init_obs, env.env_properties)
    
    actions = jnp.ones((100,1)) * 0.5
        
    observations, states, last_state = env.sim_ahead(
        init_state,
        actions=actions,
        env_properties=env.env_properties,
        obs_stepsize=env.tau,
        action_stepsize=env.tau,
    )
    plt.plot(states.physical_state.deflection, "b")

plt.hlines(+2.4, xmin=-5, xmax=actions.shape[0] + 5, color="r")
plt.hlines(-2.4, xmin=-5, xmax=actions.shape[0] + 5, color="r")
plt.grid()
plt.show()

new_env_properties = env.EnvProperties(
    physical_normalizations=env.env_properties.physical_normalizations,
    action_normalizations=env.env_properties.action_normalizations,
    static_params=env.StaticParams(mu_p=0.002, mu_c=0.5, l=50, m_p=0.1, m_c=1, g=9.81)
)

for init_omega in jnp.arange(-1, 1, 0.1):
    init_obs = jnp.array([0, 0, 0., init_omega])
    init_state = env.generate_state_from_observation(init_obs, env.env_properties)
    
    actions = jnp.ones((100,1)) * 0.5
        
    observations, states, last_state = env.sim_ahead(
        init_state,
        actions=actions,
        env_properties=new_env_properties,
        obs_stepsize=env.tau,
        action_stepsize=env.tau,
    )
    plt.plot(states.physical_state.deflection, "b")

plt.hlines(+2.4, xmin=-5, xmax=actions.shape[0] + 5, color="r")
plt.hlines(-2.4, xmin=-5, xmax=actions.shape[0] + 5, color="r")
plt.grid()
plt.show()

In [ ]:
observations, states, last_state = env.sim_ahead(
    init_state,
    actions=actions,
    env_properties=new_env_properties,
    obs_stepsize=env.tau,
    action_stepsize=env.tau,
)
env.StaticParams(mu_p=0.002, mu_c=0.5, l=50, m_p=0.1, m_c=1, g=9.81)

In [ ]:
def loss_function(static_params, true_observations, actions, env):

    new_env_properties = env.EnvProperties(
        physical_normalizations=env.env_properties.physical_normalizations,
        action_normalizations=env.env_properties.action_normalizations,
        static_params=static_params,
    )
  
    init_obs = true_observations[0]
    init_state = env.generate_state_from_observation(init_obs, new_env_properties)
    
    pred_observations, states, last_state = env.sim_ahead(
        init_state,
        actions=actions,
        env_properties=new_env_properties,
        obs_stepsize=env.tau,
        action_stepsize=env.tau,
    )

    return jnp.mean((pred_observations - true_observations)**2)

In [ ]:
parameter_gradient = eqx.filter_grad(loss_function)

In [ ]:
parameter_gradient(
    env.StaticParams(mu_p=jnp.array(0.002), mu_c=jnp.array(0.5), l=jnp.array(0.1), m_p=jnp.array(0.1), m_c=jnp.array(1.), g=jnp.array(9.81)),
    observations,
    actions,
    env,
)

- no gradient :( why though? **because the inputs where leaves and not jax arrays**
- some operation likely demolishes our ability to differentiate through the env
- differentiating wrt the actions still works, also for the observations, so why not for the static params?

In [ ]:
loss_function(
    env.StaticParams(mu_p=0.002, mu_c=0.5, l=0.5, m_p=0.1, m_c=1, g=9.81),
    observations,
    actions,
    env,
)

- consider a very small tolerance on the solver?